# Audio Player — NISQA Corpus

Listen to audio files from the NISQA corpus.  
Three modes: **random**, **by folder**, or **by filename**.

In [1]:
import random
from pathlib import Path

import IPython.display as ipd

from asa.data import load_audio, TARGET_SR

CORPUS_ROOT = Path("../data/raw/NISQA_Corpus")

# All available subsets
SUBSETS = sorted([d.name for d in CORPUS_ROOT.iterdir() if d.is_dir()])
print("Available subsets:")
for s in SUBSETS:
    print(f"  • {s}")

Available subsets:
  • NISQA_TEST_FOR
  • NISQA_TEST_LIVETALK
  • NISQA_TEST_P501
  • NISQA_TRAIN_LIVE
  • NISQA_TRAIN_SIM


In [2]:
def _all_wavs(subset: str | None = None) -> list[Path]:
    """Collect all .wav files, optionally filtered to a single subset."""
    roots = [CORPUS_ROOT / subset] if subset else [CORPUS_ROOT / s for s in SUBSETS]
    files = []
    for root in roots:
        for sub in ("deg", "ref"):
            d = root / sub
            if d.is_dir():
                files.extend(d.glob("*.wav"))
    return files


def play(path: Path):
    """Load and play a single audio file."""
    waveform = load_audio(str(path), target_sr=TARGET_SR)
    duration = len(waveform) / TARGET_SR
    print(f"File     : {path.name}")
    print(f"Subset   : {path.parts[-3]}")
    print(f"Type     : {path.parts[-2]}")
    print(f"Duration : {duration:.1f}s")
    
    # Load metadata if available
    subset = path.parts[-3]
    csv_file = CORPUS_ROOT / subset / f"{subset}_file.csv"
    if csv_file.exists():
        import pandas as pd
        df = pd.read_csv(csv_file)
        row = df[(df['filename_deg'] == path.name) | (df['filename_ref'] == path.name)]
        if not row.empty:
            disp_df = row[['mos', 'noi', 'dis', 'col', 'loud']].copy()
            from IPython.display import display
            display(disp_df)
        else:
            print("Metadata : None")
    
    ipd.display(ipd.Audio(waveform, rate=TARGET_SR))


---
## Option 1 — Random sample

Pick a random file from the entire corpus (or from a specific subset).

In [3]:
# Set to a subset name to restrict, or None for the full corpus
SUBSET = None  # e.g. "NISQA_TEST_FOR", "NISQA_TRAIN_SIM", ...

files = _all_wavs(SUBSET)
chosen = random.choice(files)
play(chosen)

/Users/carlschmidt-svejstrup/code/dtu/automatic-speech-assessment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


File     : c02928_1_1332_2_7_001-ch6-speaker_seg103.wav
Subset   : NISQA_TRAIN_SIM
Type     : deg
Duration : 9.0s


,mos,noi,dis,col,loud
2927,2.8,2.744436,3.45141,2.23681,3.752341


---
## Option 2 — Browse a specific folder

Pick a subset and listen to the first N files.

In [4]:
SUBSET = "NISQA_TEST_FOR"  # change to any subset
N = 3

files = _all_wavs(SUBSET)
random.shuffle(files)

for f in files[:N]:
    play(f)
    print()

File     : for_cnv_f_0043_01.wav
Subset   : NISQA_TEST_FOR
Type     : ref
Duration : 6.1s


,mos,noi,dis,col,loud
110,1.84375,2.254927,3.234082,2.446657,1.626635



File     : for_cnv_f_0026_02.wav
Subset   : NISQA_TEST_FOR
Type     : ref
Duration : 11.3s


,mos,noi,dis,col,loud
154,2.096774,2.601639,2.695855,2.327616,3.153903



File     : for_cnv_f_0036_01.wav
Subset   : NISQA_TEST_FOR
Type     : ref
Duration : 8.3s


,mos,noi,dis,col,loud
6,3.65625,3.285788,4.106004,3.475223,4.128798


---
## Option 3 — Play a specific file by name

Give a filename (e.g. `c00056_for_cnv_m_0067_02.wav`) and it will be found
automatically across all subsets.

In [5]:
FILENAME = "c00025_for_cnv_f_0004_01.wav"  # change this

matches = list(CORPUS_ROOT.rglob(FILENAME))

if not matches:
    print(f"Not found: {FILENAME}")
else:
    for m in matches:
        play(m)
        print()

File     : c00025_for_cnv_f_0004_01.wav
Subset   : NISQA_TEST_FOR
Type     : deg
Duration : 7.7s


,mos,noi,dis,col,loud
98,3.115385,4.134363,4.283413,3.803323,1.706978
